# Figure 3.2 — what the three plots show

![Figure 3.2: three piecewise linear functions](assets/figure-3-2-piecewise-linear-functions.png)

> **Figure source:** Figure 3.2 from [*Understanding Deep Learning*](../chapter-03-shallow-neural-networks.pdf), book page 26, by Simon J. D. Prince (MIT Press). The supplied screenshot is included unchanged for noncommercial study under [CC BY-NC-ND 4.0](../../../LICENSES/CC-BY-NC-ND-4.0.txt).

All three panels were produced by the **same neural-network architecture**:

$$y = \phi_0 + \sum_{d=1}^{3} \phi_d\,\operatorname{ReLU}(\theta_{d0}+\theta_{d1}x).$$

The architecture—one input, three hidden ReLU units, and one output—does not change. Panels (a), (b), and (c) use different values for the ten parameters $\phi_0,\ldots,\phi_3$ and $\theta_{10},\theta_{11},\ldots,\theta_{30},\theta_{31}$. This illustrates a **family of functions**: one parameterized formula that can produce many different input/output relationships.

Each plotted function consists of connected straight segments. The places where two segments meet and the slope changes are called **joints**, **breakpoints**, or **knots**. Three hidden ReLU units can contribute at most three joints, which divide the input axis into at most four linear regions. A joint may fall outside the displayed interval, and joints can coincide, so fewer than four visible regions are also possible.

The panels differ in three main ways:

- **Joint positions:** the bends occur at different input values.
- **Regional slopes:** each straight segment may rise, fall, or remain flat.
- **Overall height:** the entire function can be shifted vertically.

For hidden unit $d$, its ReLU changes state when

$$\theta_{d0}+\theta_{d1}x=0,$$

so, when $\theta_{d1}\neq 0$, its joint is located at

$$x_{\text{joint},d}=-\frac{\theta_{d0}}{\theta_{d1}}.$$

The $\theta$ parameters therefore help determine where units switch on or off and what slopes they contribute. The $\phi_d$ parameters scale those contributions, while $\phi_0$ shifts the final function vertically.

A flat segment, such as the left part of panel (b) or the right part of panel (c), is still a linear region—it simply has slope zero. The colored curves are not data points joined by hand; they are the continuous functions computed by the network for every value of $x$.


# Piecewise linear functions — the underlying theory

## Meaning of ‘piecewise linear’

**Piecewise** means that the input domain is divided into separate regions. **Linear** means that within each region the output follows a straight-line rule. For a scalar input, a piecewise linear function has the form

$$f(x)=\begin{cases}a_1x+b_1 & x \text{ is in region 1},\\a_2x+b_2 & x \text{ is in region 2},\\\vdots & \vdots\end{cases}$$

Each region has its own constant slope $a_k$ and intercept $b_k$. The book uses ‘linear’ for rules of the form $ax+b$; in stricter mathematical terminology, a rule with $b\neq0$ is **affine**, so these networks may also be called piecewise affine.

A simple example is the absolute-value function:

$$|x|=\begin{cases}-x & x<0,\\x & x\geq0.\end{cases}$$

It is a straight line on each side of zero, but it is not one straight line globally because its slope changes from $-1$ to $+1$ at the joint $x=0$.

## How one ReLU creates a joint

The ReLU activation is

$$\operatorname{ReLU}(z)=\max(0,z)=\begin{cases}0 & z<0,\\z & z\geq0.\end{cases}$$

For $h_d(x)=\operatorname{ReLU}(\theta_{d0}+\theta_{d1}x)$, one side of the threshold is flat at zero and the other side follows the line $\theta_{d0}+\theta_{d1}x$. The threshold is a hinge: crossing it changes whether that hidden unit contributes to the output.

Because both ReLU branches equal zero at the threshold, the function remains continuous—the segments meet without a jump—but the slope can change abruptly, creating a corner.

## Why a sum of ReLUs is still piecewise linear

Within any interval that contains no joint, every hidden unit has a fixed state:

- an **inactive** unit contributes zero;
- an **active** unit contributes its linear expression.

If $A$ is the set of active units in one interval, then

$$\begin{aligned}y&=\phi_0+\sum_{d\in A}\phi_d(\theta_{d0}+\theta_{d1}x)\\&=\left(\phi_0+\sum_{d\in A}\phi_d\theta_{d0}\right)+\left(\sum_{d\in A}\phi_d\theta_{d1}\right)x.\end{aligned}$$

The expression on the final line is just intercept plus slope times $x$, so it is linear within that interval. When $x$ crosses a joint, the active set $A$ changes. The formula is still linear in the next interval, but it has a different slope and intercept. That is exactly what **piecewise linear** means.

The slope inside one region is

$$\frac{dy}{dx}=\sum_{d\in A}\phi_d\theta_{d1}.$$

It stays constant until a ReLU changes state. At that joint, one term is added to or removed from the slope.

## A concrete three-joint example

Consider

$$f(x)=\operatorname{ReLU}(x)-2\operatorname{ReLU}(x-1)+\operatorname{ReLU}(x-2).$$

Evaluating the active ReLUs in each interval gives

$$f(x)=\begin{cases}0 & x<0,\\x & 0\leq x<1,\\2-x & 1\leq x<2,\\0 & x\geq2.\end{cases}$$

The result is flat, then rises, then falls, then becomes flat again. Every region is a line, but the complete function forms a nonlinear tent shape. This is the same mechanism behind the changing slopes in Figure 3.2.

## Locally linear, globally nonlinear

A ReLU network is linear **inside each activation region**, but the full input/output function is generally nonlinear because the active units—and therefore the linear rule—change with the input. ‘Made of straight pieces’ does not mean ‘equivalent to one straight line.’ The joints are precisely where the network gains nonlinear behavior.

This also explains why removing the activation functions would be a problem. A weighted sum of linear functions is still one linear function, regardless of the number of hidden units. ReLU introduces input-dependent switches that let different linear rules apply in different regions.

## Why more hidden units increase capacity

For a one-dimensional input, each ReLU hidden unit can add a joint and therefore another linear region. More regions let the network follow a curved target more closely using many short straight segments—similar to approximating a circle with a polygon containing more and more sides. This is the intuition behind the universal approximation result discussed later in the chapter.

The theorem says that a sufficiently wide shallow network can approximate a continuous function on a bounded domain as accurately as desired. It does **not** say that training will automatically find the required parameters, that the approximation will be efficient, or that the network will behave well outside the training domain.

## Extension to multiple input dimensions

With one input, joints are points that divide a line into intervals. With multiple inputs, each ReLU threshold is a hyperplane that divides the input space into regions. In two dimensions the regions look like polygons; in higher dimensions they are polyhedral regions. Within each region, the network still computes an affine function.

Deep ReLU networks are also piecewise linear, but composing layers can create a very large number of regions. Depth allows later layers to repeatedly split and rearrange regions created by earlier layers.

## Mental model

Think of a flexible ruler made from straight sections connected by hinges. Each ReLU can introduce a hinge, the $\theta$ parameters position it, and the $\phi$ parameters control how much it changes the final slope. The network bends at the hinges but remains straight between them.


# Why the “fourth slope” is not independent

![Figure 3.3: hidden-unit slope contributions](assets/figure-3-3-hidden-unit-slope-contributions.png)

> **Figure source:** Figure 3.3 from [*Understanding Deep Learning*](../chapter-03-shallow-neural-networks.pdf), book page 28, by Simon J. D. Prince (MIT Press). The supplied screenshot is included unchanged for noncommercial study under [CC BY-NC-ND 4.0](../../../LICENSES/CC-BY-NC-ND-4.0.txt).

Each hidden unit supplies just one possible slope contribution. When unit $d$ is active, its contribution is

$$s_d=\phi_d\theta_{d1};$$

when it is inactive, its contribution is zero. The output slope in any region is therefore the **sum of the $s_d$ values for the units active in that region**. The offset $\phi_0$ changes height but not slope.

In the displayed example, reading the four regions from left to right gives:

| Region | Active units | Output slope |
|---|---|---|
| 1 | $h_3$ | $m_1=s_3$ |
| 2 | $h_1,h_3$ | $m_2=s_1+s_3$ |
| 3 | $h_1,h_2,h_3$ | $m_3=s_1+s_2+s_3$ |
| 4 | $h_1,h_2$ | $m_4=s_1+s_2$ |

There are four regional slopes but only three adjustable contributions, $s_1,s_2,s_3$. The four slopes therefore cannot be chosen independently. Here, for example,

$$m_3=m_1+m_4.$$

Once three contributions are fixed, the remaining regional slope is automatically determined. If a region has **all units inactive**, its output is only the constant $\phi_0$, so its slope is zero. Otherwise, its slope is a sum of the active hidden-unit contributions.

> **Important:** “the fourth” does not necessarily mean the rightmost region. It means that after three independent slope contributions have been chosen, one of the four regional slopes must be dependent on them.

**In short:** three ReLU units provide three slope-change controls, so they may create four regions, but not four freely adjustable slopes.
